## Load env variables and create client

In [5]:
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

## Helper functions

In [6]:
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [9]:
import json

def show_response(response):
    """Messageオブジェクトをテキストで見やすく表示"""

    # メタ情報
    print(f"{'─'*60}")
    print(f"  Model      : {response.model}")
    print(f"  Stop reason: {response.stop_reason}")
    print(f"  Tokens     : input={response.usage.input_tokens}, output={response.usage.output_tokens}")
    print(f"{'─'*60}")

    # コンテンツブロック
    for i, block in enumerate(response.content):
        if block.type == "text":
            print(block.text)
        elif block.type == "tool_use":
            print(f"\n[Id    : {block.id}]")
            print(f"[Tool  : {block.name}]")
            print(json.dumps(block.input, indent=2, ensure_ascii=False))

    print(f"{'─'*60}")

In [8]:
web_search_scheme = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5
}

## Execute

In [12]:
messages = []
add_user_message(
    messages,
    """
    What's the best excercise for gaining leg muscle?
    answer with the name of the excercise and the website that has the best explanation of how to do it.
    """,
)
response = chat(messages,tools=[web_search_scheme])
print(response)
show_response(response)

Message(id='msg_0141Yw8NnxRbGrrtRtmapnmi', content=[ServerToolUseBlock(id='srvtoolu_01TUeNR9zwBgG9ryweQbSeYo', input={'query': 'best exercise for leg muscle growth'}, name='web_search', type='server_tool_use', caller={'type': 'direct'}), WebSearchToolResultBlock(content=[WebSearchResultBlock(encrypted_content='EtIZCioIDRgCIiQ2N2M4MTBiNS1hYWM1LTQ2ODYtYjNiNy0xYmRiODY2ZmM3YmMSDFCwBdFiaUHJ9LbJGhoMo54II2Zr7Mru8j9pIjAg+Pt/U5AgnnrkcNLsmPJVcUFw2ZrSJ9jCxxSoZNE4QdLT19Fw4aok+bQ1uYD8QAUq1RiUb+BQ3Fs5p6M2x8C59S/Mjgu4e9jVWFyiZDDznKa/sD+cH0N0ZqWAe9zNRunoGIHnXDkAwV/NRBAmvK04cOzs3POkw+7jCPHtbpLVx7aNgresEc25ExuHREeJSiQPZdMEveI4dW7SC0hMWi7bH+zYNjalGR6p1HLvi1nFKHb+ImwwiVsCYsXGHT4fpsaWmSwL4MC9SdaSelQRglqefYKCFpb0qEAJ4xfYRgpJqU3JGcTDbbfR33pw/t3Ka3Xcf1+rwe3jXchqc18sTizqYPChk+mMdJprdGV8I/6zAsFgjW3w2Y2wtAn/1qQDGJtN0bo6AdPAe4Uz+5K0Pga3jaS1iRgRUf41BZ9azql3NCloxmu3ceJjKttTUe02xDNkYcKvEDBQxeKiQZqmgf+1f6JMGemJvsXgvSBf0di8G74yf0yMXiCvuXDUJlZC9nNe74idfsQDHUWt0Q0WnWonpKGJIh5Ed6WoR9SXZAfLoPuXyoqM1oU4NOS51faoIQIMb3X2nZMq

In [16]:
web_search_scheme_nih = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"]
}

In [17]:
messages = []
add_user_message(
    messages,
    """
    What's the best excercise for gaining leg muscle?
    answer with the name of the excercise and the website that has the best explanation of how to do it.
    """,
)
response = chat(messages,tools=[web_search_scheme_nih])
print(response)
show_response(response)

Message(id='msg_01RY9j6ZJCMeVXaLJpNpL6qM', content=[ServerToolUseBlock(id='srvtoolu_01T1VfeFR5WwhcQxqBCtZaoQ', input={'query': 'best exercise for leg muscle gain'}, name='web_search', type='server_tool_use', caller={'type': 'direct'}), WebSearchToolResultBlock(content=[WebSearchResultBlock(encrypted_content='EtwhCioIDRgCIiQ2N2M4MTBiNS1hYWM1LTQ2ODYtYjNiNy0xYmRiODY2ZmM3YmMSDK/at+M5pVNLGNxIJRoMX97bWzGX5MleH0/5IjCe2HfhUAV4PlSsHyD1WCk5IQK2B9xim49DYhH6bEWqLUwlIxN/bTHcq816h+YA35oq3yB3a7QG3pCGQh2YqlKT7l8UgRR6VyxpOIwQfSgvwmwMGfH4ij5zii0q+CXcd4Dd9DCe21J/WQMR13ENLCS8XzYth3sFJMpiOwes7UT/q87wOozOAe/oHcfOBfPsCgon/rMXEer6llZBpApgIFQxSMyjMVhpL4gYbMYVqsSWUZpZZm5E63VHjfmYCltEkfdfCQ5JYGpBkCIBLTQioV4wXbIQr30ofC456sTI+IOWY+JbOA0hIqDMAhmD42OmsDHVoPa3Mag+pN87d5yBcL3QPng8PsXdoL8fYgTPRK0yHqK21PHUpYSy6m7Jqzxe48NIsd+dd/J9yvAiZw5lOaW5hWVsUYSOdc3xloMGRRVGADebny+Xlckfy1hncPRkEdtIsOb2IlR54RvkL1dgbnual+uj4pRA5u/hMTnjg/RbYwV5HYk5DgdVDGgM3QhrlOtqwfUKQcT/CSj4VCI21MiaCAqeXYthhci7cRmi4Pp0+dJxO19L8eIliZMXQ7hNGmtcVUT11dX2DP